# Tahap 2: Pre-training Spasial — Data Pipeline FER2013

**Judul Skripsi:** Deteksi Mikro-Ekspresi Wajah

Notebook ini menyiapkan **data pipeline** untuk pre-training spasial menggunakan dataset macro-expression **FER2013**:

1. **Konfigurasi** — path dataset & hyperparameter
2. **Transforms** — augmentasi ringan (flip + rotasi) dan normalisasi standar ImageNet
3. **Dataset Loading** — `torchvision.datasets.ImageFolder` (struktur sub-folder per kelas emosi)
4. **DataLoader** — batch loader untuk training & testing
5. **Sanity Check** — visualisasi 1 batch gambar beserta label emosinya

> **Catatan penting:** Gambar FER2013 berformat grayscale 48×48 piksel. Karena backbone pre-trained ImageNet menerima input RGB 3-channel, gambar dikonversi dengan `transforms.Grayscale(num_output_channels=3)` sebelum normalisasi.

In [ ]:
# =============================================================================
# 1. KONFIGURASI DIREKTORI & HYPERPARAMETER
# =============================================================================
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

# --- Path dataset (ubah di sini jika lokasi/struktur dataset berganti) ---
# Struktur yang diharapkan: <DIR>/<nama_kelas>/<gambar>.jpg
DATASET_ROOT = Path('dataset 1/fer2013/fer2013')   # root dataset FER2013 lokal
TRAIN_DIR = DATASET_ROOT / 'Training'              # folder data train
TEST_DIR = DATASET_ROOT / 'Test'                   # folder data test (opsional)

# --- Hyperparameter data pipeline ---
BATCH_SIZE = 32          # jumlah gambar per batch
IMG_SIZE = 224           # ukuran input backbone pre-trained (224x224)
NUM_WORKERS = 2          # worker paralel DataLoader (set 0 jika bermasalah di macOS/Jupyter)
TEST_SPLIT_RATIO = 0.2   # proporsi test jika folder Test belum tersedia (fallback split 80/20)
RANDOM_SEED = 42         # seed agar pembagian train/test reproducible

# --- Normalisasi standar ImageNet (wajib untuk transfer learning) ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Perangkat komputasi (dipakai di tahap training nanti)
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

# Validasi awal: pastikan folder train ada sebelum lanjut
if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(
        f'Folder train tidak ditemukan: {TRAIN_DIR.resolve()}\n'
        'Sesuaikan variabel DATASET_ROOT / TRAIN_DIR di atas.'
    )

print(f'Train dir : {TRAIN_DIR.resolve()}')
print(f'Test dir  : {TEST_DIR.resolve()} {"(ada)" if TEST_DIR.is_dir() else "(TIDAK ada — fallback split dari train)"}')
print(f'Device    : {DEVICE}')

## 2. Definisi Transforms (Augmentasi & Normalisasi)

| Transform | Train | Test | Alasan |
|-----------|:-----:|:----:|--------|
| `Grayscale(3ch)` | ✓ | ✓ | FER2013 grayscale → duplikasi ke 3 channel agar cocok dengan backbone ImageNet |
| `Resize 224×224` | ✓ | ✓ | Ukuran input standar model pre-trained |
| `RandomHorizontalFlip` | ✓ | — | Augmentasi ringan: wajah simetris, flip tidak mengubah label emosi |
| `RandomRotation ±10°` | ✓ | — | Augmentasi ringan: simulasi kemiringan kepala natural, mencegah overfitting |
| `ToTensor` | ✓ | ✓ | Konversi PIL → Tensor float [0,1] |
| `Normalize (ImageNet)` | ✓ | ✓ | Menyamakan distribusi input dengan data pre-training backbone |

Augmentasi **tidak** diterapkan pada data test agar evaluasi konsisten dan deterministik.

In [ ]:
# =============================================================================
# 2. TRANSFORMS — AUGMENTASI (TRAIN) & NORMALISASI (TRAIN + TEST)
# =============================================================================

# --- Transforms untuk data TRAIN (dengan augmentasi ringan) ---
train_transforms = transforms.Compose([
    # FER2013 grayscale 1-channel -> duplikasi jadi 3-channel (kompatibel backbone RGB)
    transforms.Grayscale(num_output_channels=3),
    # Samakan ukuran gambar dengan input backbone pre-trained
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # Augmentasi: flip horizontal acak (peluang 50%) — wajah tetap valid saat dicerminkan
    transforms.RandomHorizontalFlip(p=0.5),
    # Augmentasi: rotasi acak maksimum ±10 derajat — simulasi kemiringan kepala
    transforms.RandomRotation(degrees=10),
    # Konversi PIL Image -> Tensor float dengan rentang [0, 1]
    transforms.ToTensor(),
    # Normalisasi channel RGB memakai statistik ImageNet
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# --- Transforms untuk data TEST (TANPA augmentasi — evaluasi harus deterministik) ---
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('train_transforms:')
print(train_transforms)
print('\ntest_transforms:')
print(test_transforms)

## 3. Dataset Loading — `ImageFolder`

`torchvision.datasets.ImageFolder` otomatis membaca struktur `folder/kelas/gambar` dan memberi label integer per kelas (urut alfabetis).

**Fallback split:** jika folder `Test` belum tersedia (kondisi dataset saat ini), data train dibagi otomatis **80% train / 20% test** secara acak dengan seed tetap (`RANDOM_SEED`) agar hasil reproducible. Dua instance `ImageFolder` dibuat dari folder yang sama — masing-masing dengan transforms train dan test — lalu dipisah berdasarkan indeks yang sama supaya tidak ada kebocoran data (*data leakage*).

In [ ]:
# =============================================================================
# 3. DATASET LOADING — ImageFolder (+ fallback split 80/20 jika Test belum ada)
# =============================================================================

if TEST_DIR.is_dir():
    # --- Kasus ideal: folder train & test terpisah ---
    train_dataset = datasets.ImageFolder(str(TRAIN_DIR), transform=train_transforms)
    test_dataset = datasets.ImageFolder(str(TEST_DIR), transform=test_transforms)
    class_names = train_dataset.classes
else:
    # --- Fallback: folder Test belum ada -> split 80/20 dari data train ---
    # Dua ImageFolder dari folder yang SAMA, tapi transforms berbeda:
    # augmentasi hanya boleh aktif pada subset train.
    base_train = datasets.ImageFolder(str(TRAIN_DIR), transform=train_transforms)
    base_test = datasets.ImageFolder(str(TRAIN_DIR), transform=test_transforms)
    class_names = base_train.classes

    # Acak indeks dengan seed tetap agar pembagian selalu sama (reproducible)
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    indices = torch.randperm(len(base_train), generator=generator).tolist()
    split_point = int(len(indices) * (1 - TEST_SPLIT_RATIO))

    # Subset memakai indeks yang saling lepas -> tidak ada data leakage
    train_dataset = Subset(base_train, indices[:split_point])
    test_dataset = Subset(base_test, indices[split_point:])

# --- Informasi dataset (sanity check jumlah data & kelas) ---
print(f'Jumlah gambar Train : {len(train_dataset)}')
print(f'Jumlah gambar Test  : {len(test_dataset)}')
print(f'Jumlah kelas emosi  : {len(class_names)}')
print(f'Nama kelas          : {class_names}')

## 4. PyTorch DataLoader

- `train_loader`: `shuffle=True` — urutan data diacak tiap epoch agar model tidak menghafal urutan.
- `test_loader`: `shuffle=False` — evaluasi harus konsisten antar run.
- `num_workers`: jumlah proses paralel pembaca data. Jika DataLoader macet di macOS/Jupyter, ubah `NUM_WORKERS = 0` di cell konfigurasi.

In [ ]:
# =============================================================================
# 4. DATALOADER — BATCH LOADER UNTUK TRAINING & TESTING
# =============================================================================

# DataLoader train: shuffle aktif agar urutan sampel acak setiap epoch
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

# DataLoader test: tanpa shuffle agar evaluasi deterministik
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f'Batch size          : {BATCH_SIZE}')
print(f'Jumlah batch Train  : {len(train_loader)}')
print(f'Jumlah batch Test   : {len(test_loader)}')

# Sanity check: ambil 1 batch dan cek dimensi tensor
sample_images, sample_labels = next(iter(train_loader))
print(f'Shape batch gambar  : {tuple(sample_images.shape)}  (N, C, H, W)')
print(f'Shape batch label   : {tuple(sample_labels.shape)}')

## 5. Visualisasi Batch (Sanity Check)

Menampilkan 16 gambar pertama dari 1 batch dalam bentuk grid. Tensor harus di-**de-normalisasi** dulu (`img = img * std + mean`) agar warna kembali normal saat ditampilkan — jika tidak, gambar terlihat gelap/aneh karena masih dalam skala normalisasi ImageNet.

In [ ]:
# =============================================================================
# 5. FUNGSI VISUALISASI BATCH (SANITY CHECK)
# =============================================================================

def imshow_batch(loader, classes, num_images: int = 16):
    """
    Menampilkan grid gambar dari 1 batch DataLoader beserta label emosinya.

    Parameters
    ----------
    loader : DataLoader
        DataLoader sumber (train_loader / test_loader).
    classes : list[str]
        Daftar nama kelas emosi (urutan sesuai label integer).
    num_images : int
        Jumlah gambar yang ditampilkan (8–16 disarankan).
    """
    # Ambil tepat 1 batch dari loader
    images, labels = next(iter(loader))

    # Batasi jumlah gambar yang divisualisasikan
    images = images[:num_images]
    labels = labels[:num_images]

    # --- De-normalisasi: img = img * std + mean (kebalikan dari Normalize) ---
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)   # reshape agar broadcast per-channel
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    images = images * std + mean
    images = torch.clamp(images, 0.0, 1.0)             # jaga rentang piksel [0, 1]

    # Susun gambar menjadi grid (4 kolom per baris)
    grid = make_grid(images, nrow=4, padding=2)

    # make_grid menghasilkan (C, H, W) -> matplotlib butuh (H, W, C)
    grid_np = grid.permute(1, 2, 0).numpy()

    # Nama label emosi tiap gambar sebagai judul grid
    label_names = [classes[int(lbl)] for lbl in labels]
    title_lines = [
        ' | '.join(label_names[i:i + 4]) for i in range(0, len(label_names), 4)
    ]

    plt.figure(figsize=(10, 10))
    plt.imshow(grid_np)
    plt.title('\n'.join(title_lines), fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


# Jalankan sanity check pada train_loader (label urut kiri->kanan, baris per baris)
imshow_batch(train_loader, class_names, num_images=16)

# Tahap 3: Baseline Model ResNet-18 — Pre-training Spasial

Bagian ini melatih baseline CNN **ResNet-18 pre-trained ImageNet** menggunakan DataLoader FER2013 yang sudah dibuat pada cell sebelumnya.

Alur training setiap epoch:

1. **Fase train** — aktifkan `model.train()`, hitung gradien, dan perbarui bobot.
2. **Fase evaluasi** — aktifkan `model.eval()` dan nonaktifkan gradien dengan `torch.no_grad()`.
3. **Scheduler** — turunkan learning rate jika akurasi validasi tidak membaik.
4. **Checkpoint** — simpan `state_dict` model dengan akurasi validasi terbaik ke `resnet18_best_baseline.pth`.

> Notebook data pipeline sebelumnya menggunakan nama `test_loader` dan `class_names`. Cell berikut hanya membuat alias `val_loader` dan `classes` ke objek tersebut—**tidak membuat ulang Dataset atau DataLoader**.

In [ ]:
# =============================================================================
# 6. SETUP DEVICE, MODEL RESNET-18, LOSS, OPTIMIZER, DAN SCHEDULER
# =============================================================================
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
from tqdm.auto import tqdm

# --- Kompatibilitas nama dengan data pipeline pada cell sebelumnya ---
# Ini hanya alias ke objek yang sudah ada, bukan mendefinisikan ulang DataLoader.
if 'val_loader' not in globals():
    if 'test_loader' not in globals():
        raise NameError('val_loader/test_loader belum tersedia. Jalankan semua cell data pipeline terlebih dahulu.')
    val_loader = test_loader

if 'classes' not in globals():
    if 'class_names' not in globals():
        raise NameError('classes/class_names belum tersedia. Jalankan semua cell data pipeline terlebih dahulu.')
    classes = class_names

# --- Deteksi perangkat komputasi secara otomatis ---
if torch.cuda.is_available():
    device = torch.device('cuda')       # GPU NVIDIA
elif torch.backends.mps.is_available():
    device = torch.device('mps')        # GPU Apple Silicon
else:
    device = torch.device('cpu')        # Fallback CPU

print(f'Device yang digunakan: {device}')

# Jumlah output classifier harus sama dengan jumlah kelas emosi FER2013
num_classes = len(classes)
print(f'Jumlah kelas emosi  : {num_classes}')
print(f'Nama kelas          : {classes}')

# Muat ResNet-18 dengan bobot hasil pre-training ImageNet-1K
model = resnet18(weights='IMAGENET1K_V1')

# Ambil jumlah fitur input pada fully connected layer bawaan ResNet-18
num_features = model.fc.in_features

# Ganti classifier terakhir agar output sesuai jumlah kelas emosi
model.fc = nn.Linear(num_features, num_classes)

# Pindahkan seluruh parameter model ke perangkat komputasi aktif
model = model.to(device)

# CrossEntropyLoss sesuai untuk klasifikasi multi-kelas dengan label integer
criterion = nn.CrossEntropyLoss()

# Adam memperbarui seluruh parameter model; weight decay membantu regularisasi
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4,
)

# Turunkan learning rate menjadi 50% jika validation accuracy stagnan selama 2 epoch
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2,
)

print('\nModel ResNet-18 siap dilatih.')
print(model.fc)

In [ ]:
# =============================================================================
# 7. FUNGSI TRAINING & VALIDASI RESNET-18
# =============================================================================
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    num_epochs: int = 10,
    checkpoint_path: str = 'resnet18_best_baseline.pth',
):
    """
    Melatih dan mengevaluasi model klasifikasi selama beberapa epoch.

    Model dengan validation accuracy tertinggi otomatis disimpan sebagai checkpoint.

    Parameters
    ----------
    model : torch.nn.Module
        Model CNN yang akan dilatih.
    train_loader, val_loader : DataLoader
        DataLoader yang sudah dibuat pada tahap data preprocessing.
    criterion : torch.nn.Module
        Fungsi loss, yaitu CrossEntropyLoss.
    optimizer : torch.optim.Optimizer
        Algoritma optimasi, yaitu Adam.
    scheduler : torch.optim.lr_scheduler.ReduceLROnPlateau
        Scheduler berdasarkan validation accuracy.
    num_epochs : int
        Jumlah epoch training.
    checkpoint_path : str
        Lokasi penyimpanan bobot model terbaik.

    Returns
    -------
    model : torch.nn.Module
        Model dengan bobot terbaik berdasarkan validation accuracy.
    history : dict
        Riwayat loss dan accuracy untuk kebutuhan grafik/evaluasi selanjutnya.
    """
    # -inf memastikan checkpoint pertama selalu tersimpan, sekalipun akurasi awal 0%.
    best_val_accuracy = float('-inf')

    # Simpan riwayat metrik agar bisa diplot pada tahap evaluasi berikutnya
    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': [],
        'learning_rate': [],
    }

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch + 1}/{num_epochs}')
        print('-' * 72)

        epoch_metrics = {}

        # Dua fase per epoch: training dan evaluasi/validasi
        for phase, loader in [('train', train_loader), ('eval', val_loader)]:
            is_training = phase == 'train'

            if is_training:
                model.train()   # aktifkan mode training (BatchNorm/Dropout)
            else:
                model.eval()    # aktifkan mode evaluasi

            running_loss = 0.0
            running_correct = 0
            total_samples = 0

            # Progress bar menampilkan loss dan accuracy berjalan per batch
            progress_bar = tqdm(
                loader,
                desc='Training' if is_training else 'Validation',
                leave=False,
            )

            for images, labels in progress_bar:
                # Pindahkan batch gambar dan label ke device yang sama dengan model
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                # Bersihkan gradien lama hanya saat fase training
                if is_training:
                    optimizer.zero_grad(set_to_none=True)

                # Pada evaluasi, autograd dinonaktifkan agar lebih cepat dan hemat memori
                with torch.set_grad_enabled(is_training):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    predictions = outputs.argmax(dim=1)

                    # Backpropagation dan update bobot hanya pada fase training
                    if is_training:
                        loss.backward()
                        optimizer.step()

                batch_size = images.size(0)
                running_loss += loss.item() * batch_size
                running_correct += (predictions == labels).sum().item()
                total_samples += batch_size

                # Metrik sementara pada progress bar
                progress_bar.set_postfix(
                    loss=f'{running_loss / total_samples:.4f}',
                    acc=f'{100.0 * running_correct / total_samples:.2f}%',
                )

            # Metrik rata-rata satu fase penuh
            phase_loss = running_loss / total_samples
            phase_accuracy = running_correct / total_samples
            epoch_metrics[phase] = {
                'loss': phase_loss,
                'accuracy': phase_accuracy,
            }

        train_loss = epoch_metrics['train']['loss']
        train_accuracy = epoch_metrics['train']['accuracy']
        val_loss = epoch_metrics['eval']['loss']
        val_accuracy = epoch_metrics['eval']['accuracy']

        # Scheduler mode='max' memonitor validation accuracy
        scheduler.step(val_accuracy)
        current_lr = optimizer.param_groups[0]['lr']

        # Catat semua metrik epoch
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['learning_rate'].append(current_lr)

        # Simpan state_dict hanya ketika validation accuracy mencapai nilai terbaik baru
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), checkpoint_path)
            checkpoint_status = f'checkpoint tersimpan: {checkpoint_path}'
        else:
            checkpoint_status = 'tidak ada peningkatan checkpoint'

        # Ringkasan metrik di akhir epoch
        print(
            f'Train Loss: {train_loss:.4f} | '
            f'Train Accuracy: {train_accuracy * 100:.2f}%'
        )
        print(
            f'Val Loss  : {val_loss:.4f} | '
            f'Val Accuracy  : {val_accuracy * 100:.2f}%'
        )
        print(f'Learning Rate: {current_lr:.6f} | {checkpoint_status}')

    # Muat kembali bobot terbaik, bukan bobot dari epoch terakhir
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print('\nTraining selesai.')
    print(f'Validation accuracy terbaik: {best_val_accuracy * 100:.2f}%')
    print(f'Model terbaik dimuat dari  : {checkpoint_path}')

    return model, history

In [ ]:
# =============================================================================
# 8. JALANKAN PRE-TRAINING SPASIAL BASELINE RESNET-18
# =============================================================================
NUM_EPOCHS = 10
CHECKPOINT_PATH = 'resnet18_best_baseline.pth'

# Fungsi memakai langsung train_loader dan val_loader dari scope global
model, training_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    checkpoint_path=CHECKPOINT_PATH,
)